# Homotopy operators and high-order polar complexes

This developer tutorial walks through the construction of the high-order polar de Rham complex of R. Remmerswaal
(NMPP seminar 2026; Julia package [PolarFEEC.jl](https://gitlab.mpcdf.mpg.de/rwr/polarfeec.jl)) as implemented in
`struphy.polar.homotopy`, on the 1D spaces of Struphy `Derham` complexes and on Struphy domains:

0. 1D spaces, tensor-product and mapped forms with commuting projections,
1. the Poincaré homotopy operators and their matrices,
2. the polar complex: extraction and dual matrices, cochain/chain property, cohomology,
3. commuting polar projections: convergence and commutativity,
4. boundary-compatible homotopies and the 2D curl–curl problem ("a free two-dimensional lunch"),
5. the Maxwell spectrum on the disk,
6. the 3D total complex and the Struphy `Derham` integration,
7. the 0- and 1-form wave equations: regularity and convergence,
8. field lines and the assembly cost in the 3D `Derham` complex,
9. the 3D wave in a helical tube and the schematics of the talk.

Set `STRUPHY_TUTORIAL_FULL=1` to run the convergence studies at the resolution of the talk and of PolarFEEC.jl.

In [ ]:
import os

import matplotlib.pyplot as plt
import numpy as np
import scipy.sparse as sp

from struphy.geometry import domains
from struphy.polar.homotopy import cumulative_sum_matrix
from struphy.polar.homotopy.demos import common
from struphy.polar.homotopy.linalg import as_dense

FULL = os.environ.get("STRUPHY_TUTORIAL_FULL", "0") == "1"

## 0. Spaces, forms and commuting projections (PolarFEEC.jl examples 01–03)

The building blocks are the 1D spaces of a Struphy `Derham` complex (`DerhamSpace1D`: the feectools B-splines and
their M-splines, clamped or periodic, or the nodal trigonometric functions of `struphy.feec.trig_space`), whose
derivative is an exact coefficient map, $\tfrac{d}{dx}B_i = C_{i-1} - C_i$, so that the 1D diagram
$\tfrac{d}{dx}\pi^0 f = \pi^1 f'$ commutes:

In [ ]:
from struphy.polar.homotopy.demos import spaces

spaces.spaces_1d()
spaces.figure_spaces_1d();

Tensor products give 0-, 1- and 2-forms on the unit square with $\operatorname{curl}\circ\operatorname{grad} = 0$
exactly and projections that commute with the derivatives. The commuting and $L^2$ projections converge at order
$p + 1$ for 0-forms and $p$ for 1- and 2-forms:

In [ ]:
spaces.identities()
square_cells = (8, 16, 32, 64, 128) if FULL else (8, 16, 32)
square_errors = spaces.convergence(degrees=(1, 2, 3, 4, 5) if FULL else (1, 2, 3), cells=square_cells, verbose=False)
spaces.figure_convergence(square_errors, square_cells);

On a mapped domain (the quarter annulus `HollowCylinder(a1=1, a2=2, poc=4)`) the forms are pushed forward as scalars,
by the covariant Piola map and as densities; the projections still commute:

In [ ]:
spaces.mapped_identities()
spaces.figure_mapped_forms();

## 1. Homotopy operators

In 1D the fundamental theorem of calculus reads $P = \mathfrak P P + \partial^\dagger \partial P$ and
$A = \partial \partial^\dagger A$ with $\mathfrak P P = P(0)$ and $\partial^\dagger A = \int_0^x A$. For splines with
the derivative-scaled 1-form basis ($\tfrac{d}{dx} B_i = C_{i-1} - C_i$) the derivative is the forward difference
$\delta$ and $\partial^\dagger$ is the cumulative sum $\Sigma$, with $\delta\Sigma = I$ and
$\Sigma\delta = I - \mathbb 1 e_1^\top$:

In [ ]:
n = 6
Sigma = cumulative_sum_matrix(n).toarray()
delta = common.spline_space_1d(3, n - 2).derivative_matrix().toarray()
print("delta Sigma = I:", np.allclose(delta @ Sigma, np.eye(n)))
print("Sigma delta = I - 1 e1^T:", np.allclose(Sigma @ delta, np.eye(n + 1) - np.outer(np.ones(n + 1), np.eye(n + 1)[0])))

In 2D the Poincaré homotopies $(\operatorname{grad}^\dagger A)(x) = \int_0^1 A(tx)\cdot x\,dt$ and
$(\operatorname{curl}^\dagger B)(x) = (-y, x)^\top \int_0^1 t B(tx)\,dt$ satisfy
$P = \mathfrak P P + \operatorname{grad}^\dagger\operatorname{grad}P$,
$A = \operatorname{grad}\operatorname{grad}^\dagger A + \operatorname{curl}^\dagger\operatorname{curl}A$,
$B = \operatorname{curl}\operatorname{curl}^\dagger B$. On a disk-like mapping they are radial integrations in logical
coordinates: $G^\dagger = [I \otimes \Sigma,\ 0]$ and $C^\dagger = [0;\ I \otimes \Sigma]$.

In [ ]:
c = common.polar_complex(3, degree=3, cells=(8, 16))  # PerturbedDiskCylinder, Struphy ordering
G, C = c.spaces[0].derivative_matrix, c.spaces[1].derivative_matrix
HG, HC = c.gradient_homotopy_matrix, c.curl_homotopy_matrix
print("C C^dagger = I:", abs(C @ HC - sp.identity(HC.shape[1])).max())
print("G^dagger C^dagger = 0:", abs(HG @ HC).max())

## 2. The polar complex

Each degree is represented by an extraction matrix $E$ (its columns are the tensor-product coefficients of the polar
basis functions, central ones first) and dual matrix $F$ with $F^\top E = I$. The polar derivatives, homotopies and
masses are $\bar D_k = F_{k+1}^\top D E_k$, $h_k = F_{k-1}^\top H E_k$ and $\bar M_k = E_k^\top M E_k$.
The 1-form duals are defined by $E F^\top = \bar\Pi^1 = \operatorname{grad}\bar\Pi^0\operatorname{grad}^\dagger +
\operatorname{curl}^\dagger\bar\Pi^2\operatorname{curl}$, which makes all projections commute with the derivatives
and with the homotopies.

In [ ]:
print(c)
print("central functions (0-, 1-, 2-forms):", c.central_counts())
for k in range(3):
    print(f"F^T E = I (degree {k}):", abs(c.duals(k).T @ c.extraction(k) - sp.identity(c.dimension(k))).max())
print("cohomology:", c.cohomology_dimensions())
for k in range(3):
    m = c.dimension(k)
    contraction = np.zeros((m, m))
    if k < 2:
        contraction += as_dense(c.homotopy(k + 1) @ c.derivative(k))
    if k > 0:
        contraction += as_dense(c.derivative(k - 1) @ c.homotopy(k))
    error = np.abs(contraction - (np.eye(m) - as_dense(c.cohomology_projection(k)))).max()
    print(f"d h + h d = I - P on degree {k}: {error:.1e}")

The enrichment options of the paper: `"cochain_chain"` (default), `"cochain"` (the reduced spaces of Section 4.2,
without the compact 2-forms) and `"none"` (external functions only):

In [ ]:
for enrichment in ("cochain_chain", "cochain", "none"):
    e = common.polar_complex(3, degree=3, cells=(8, 16), enrichment=enrichment)
    print(f"{enrichment:14s} dimensions {[e.dimension(k) for k in range(3)]}, cohomology {e.cohomology_dimensions()}")

## 3. Commuting polar projections

The polar projections $\bar\pi^k = \bar\Pi^{k} \circ \pi^k$ (commuting) and the polar $L^2$ projections, for
$r_x = r_y = \mathfrak r$ (B-spline degree equal to the regularity) on the perturbed disk:

In [ ]:
from struphy.polar.homotopy.demos import projections

regularities = (1, 2, 3, 4, 5, 6) if FULL else (1, 2, 3)
cells = (4, 8, 16, 32) if FULL else (4, 8, 16)
errors_l2 = projections.convergence(regularities, cells, method="l2")
errors_commuting = projections.convergence(regularities, cells, method="commuting")
projections.figure_convergence(errors_l2, cells, title="$L^2$ projection")
projections.figure_convergence(errors_commuting, cells, title="commuting projection");

Commutativity with $\operatorname{grad}$, $\operatorname{curl}$, $\operatorname{grad}^\dagger$ and
$\operatorname{curl}^\dagger$: the commuting projections commute to round-off, the $L^2$ projections only up to the
approximation error.

In [ ]:
defects_l2 = projections.commutation(regularities, cells, method="l2")
defects_commuting = projections.commutation(regularities, cells, method="commuting")
projections.figure_commutation(defects_l2, cells, title="$L^2$ projection")
projections.figure_commutation(defects_commuting, cells, title="commuting projection");

## 4. Boundary conditions and the 2D curl–curl problem

With `boundary="dirichlet"` the relative spaces ($P = 0$, $A \times n = 0$ on $\partial\Omega$) are built. Their
homotopies are the boundary-compatible homotopies of the paper, and the degree-2 cohomology projection is the mean
projection $\mathfrak B$, so that $B = \mathfrak B B + \operatorname{curl}\operatorname{curl}^\dagger_\partial B$.
With $A = \operatorname{grad}P + \operatorname{curl}^\dagger_\partial B$ the curl–curl problem
$(\operatorname{curl}\Phi, \operatorname{curl}A) = (\Phi, J)$ reduces to **one polar 2-form mass solve**:

$$
A = h_2\,(I - \mathfrak B)\,\bar M_2^{-1} h_2^\top \ell .
$$

In [ ]:
from struphy.polar.homotopy.solvers import curl_curl_free_lunch, stiffness_solver

d = common.polar_complex(3, degree=3, cells=(6, 7), angular="trig", boundary="dirichlet")
hG, hC, B = (as_dense(a) for a in (d.homotopy(1), d.homotopy(2), d.cohomology_projection(2)))
D0, D1 = d.derivative(0).toarray(), d.derivative(1).toarray()
print("hG D0 = I:", np.abs(hG @ D0 - np.eye(D0.shape[1])).max())
print("D1 hC = I - B:", np.abs(D1 @ hC - (np.eye(D1.shape[0]) - B)).max())
load = d.mass_matrix(1) @ np.cos(np.arange(d.dimension(1)))
a = curl_curl_free_lunch(d, load)
b = stiffness_solver(d, 1).range_projection(d.homotopy(2).T @ load)
print("free lunch = saddle-point range projection:", np.abs(d.derivative(1) @ a - b).max())

### Poisson and curl–curl on a rounded square (PolarFEEC.jl example 05)

Manufactured solutions on $\Omega = \{g > 0\}$, $g = 1 - x^2 - y^2 - \varepsilon\,\mathrm{Re}((x + iy)^4)$; the
solvers work in the homotopy gauge (free Poisson: $u_h(0) = 0$; curl–curl: $h A_h = 0$).

The steps of the boundary-compatible homotopy for a zero-mean 2-form: the radial homotopy leaves a tangential trace;
the periodic primitive of this trace, extended into the outer ring, is subtracted as a gradient:

In [ ]:
from struphy.polar.homotopy.demos import schematics

schematics.boundary_homotopy();

In [ ]:
from struphy.polar.homotopy.demos import pde

solution = pde.solve_all(3, 6)
print("L2 errors (u free, u Dirichlet, curl A free, curl A Dirichlet):", pde.errors(solution))
print(pde.checks(solution))
pde.figure_poisson(solution)
pde.figure_curl_curl(solution);

In [ ]:
cells05 = pde.CELLS if FULL else (4, 6, 8, 12)
degrees05 = pde.DEGREES if FULL else (2, 3)
table = pde.convergence(cells05, degrees05)
pde.figure_convergence(table, cells05, degrees05);

## 5. The Maxwell spectrum on the unit disk (PolarFEEC.jl example 06)

The Dirichlet polar 1-forms give a spectrally correct discretization: the kernel of
$\bar C^\top \bar M_2 \bar C a = \lambda \bar M_1 a$ is exactly $\operatorname{grad}\bar V^0$, there are no spurious
modes, and the eigenvalues converge to the squared zeros of the Bessel derivatives at order $2p$.

In [ ]:
from struphy.polar.homotopy.demos import maxwell

spectrum = maxwell.study(meshes=maxwell.MESHES if FULL else ((8, 16), (16, 32)), compared=40)
maxwell.figure_spectrum(spectrum)
maxwell.figure_modes(spectrum);

## 6. The 3D total complex

In 3D the polar complex is $\bar W^m = \bigoplus_{p+q=m} \bar V^{p}_{2D} \otimes V^q_{1D}$ with
$d(V \otimes U) = d V \otimes U + (-1)^p V \otimes d U$ and the ordered homotopy
$h(V \otimes U) = h V \otimes U + (-1)^p \mathfrak P V \otimes h U$. It has the cohomology of the solid torus,
$(1, 1, 0, 0)$, and $(0, 0, 1, 1)$ with Dirichlet conditions:

In [ ]:
from struphy.polar.homotopy.complex3d import PolarComplex3D

section = domains.PerturbedDiskCylinder(R1=0.5, eps=0.1, k=2, Lz=1.0)
for boundary in ("free", "dirichlet"):
    c2 = common.polar_complex(2, degree=3, cells=(3, 8), domain=section, boundary=boundary)
    c3 = PolarComplex3D(c2, common.spline_space_1d(3, 8, periodic=True))
    print(boundary, [c3.dimension(m) for m in range(4)], c3.cohomology_dimensions())

### Integration in `Derham`

`PolarExtractionBlocksHomotopy` (in `struphy.polar.homotopy.derham_blocks`) provides the blocks of the polar operators
of `struphy.polar.linear_operators` for the total complex. Per tensor-product component, the first `n_rings` rings
hold the polar coefficients (the central polar functions and the external functions of these rings): $\mathfrak r + 2$
rings for 0-forms, $(\mathfrak r + 1, \mathfrak r + 2)$ for the two components of 2D 1-forms and $\mathfrak r + 1$ for
2D 2-forms. With these ring counts the extraction and the duals are the identity outside the polar rings, and the
derivatives map no tensor-product coefficient into the polar coefficients. The duals of the paper are used with the
non-conforming ring-0 $y$-component removed, $\tilde F = Z F$; on the coefficients of physical fields both coincide.
The commuting projectors are $\tilde F^\top \pi$, so that the polar "degrees of freedom" are the polar coefficients and
the polar inter-/histopolation matrix is the identity.

In [ ]:
from struphy.feec.psydac_derham import Derham
from struphy.io.options import DerhamOptions
from struphy.topology.grids import TensorProductGrid

options = DerhamOptions(degree=(3, 3, 1), bcs=(("free", "free"), None, None), polar_splines=True, polar_regularity=2)
derham = Derham(TensorProductGrid(num_elements=(8, 12, 2)), options, domain=domains.HollowTorus(a1=0.0, a2=0.3))
blocks = derham.ck_blocks
print("polar sets (2D):", {key: len(columns) for key, columns in blocks._sets.items()})
print("grad blocks [Hcurl comp][H1 comp] equal the 2D polar gradient:",
      np.allclose(blocks.grad_pol_to_pol[0][0].toarray(),
                  blocks.complex2d.derivative(0).toarray()[np.ix_(blocks._sets["1x"], blocks._sets["0"])]))

## 7. Wave equations: regularity and convergence (talk, Sections 1, 3 and 5)

The 0- and 1-form wave equations $\bar M v'' + \bar K v = 0$ (Dirichlet; $\bar K = (DE)^\top M (DE)$ with $D$ the
gradient or the curl) are integrated with the staggered leapfrog at $0.9$ times the stability limit. The initial 1-form
is the homotopy lift $A = h_2 (I - \mathfrak B) b$ of the projected Gaussian $b$, so that $\operatorname{curl}A = b$ up
to its mean. The talk measures the maximum relative $L^\infty$ error during the final 10% of $[0, 1]$; since the talk
does not state the reference solution, a finer mesh of the highest regularity is used here. Without
`STRUPHY_TUTORIAL_FULL=1` the study uses cubic splines, a wider pulse and $T = 0.3$. In this smooth setting all
regularities converge, and $\mathfrak r = 0$ (the largest space) has the smallest error; the stagnation of low
regularities reported in the talk concerns its narrow pulse and quintic splines (`STRUPHY_TUTORIAL_FULL=1`, which
takes hours and has not been run for this tutorial).

In [ ]:
from struphy.polar.homotopy.demos import waves

if FULL:
    wave_args = dict(regularities=(0, 1, 2, 3, 4, 5), radial_cells=(8, 16, 32), reference_cells=64, degree=5)
else:
    # the pulse of the talk (width 0.04) is not resolved on coarse meshes: use a wider one
    wave_args = dict(regularities=(0, 1, 2), radial_cells=(4, 8, 16, 32), reference_cells=64, degree=3,
                     initial=lambda x, y: waves.gaussian(x, y, width=0.15))
wave_study = waves.convergence(0, final_time=1.0 if FULL else 0.3, verbose=False, **wave_args)
waves.figure_convergence(wave_study, degree=wave_args["degree"])
waves.figure_time_traces(wave_study);

In [ ]:
one_form = waves.run_family(1, regularities=(1, 2) if not FULL else (1, 2, 3, 4), final_time=0.3,
                            **(dict(degree=5, radial_cells=8, angular_cells=16) if FULL
                               else dict(degree=3, radial_cells=6, angular_cells=12)))
waves.figure_snapshots(one_form);

`waves.WaveProblem` also takes `angular="trig"` (the trigonometric angular spaces of the talk), `enrichment="none"`
or `"cochain"` and a `space_mapping` different from the pushforward (the non-matching study).

## 8. Field lines and assembly cost in the 3D `Derham` complex

The field-line study of PolarFEEC.jl example 07 (see the user tutorial) and the assembly cost of the polar mass
matrices against the degree (the counterpart of example 08; Struphy assembles the tensor-product masses with its own
kernels and applies $E^\top M E$):

In [ ]:
from struphy.polar.homotopy.demos import performance

timings = performance.study(degrees=(1, 2, 3, 4, 5) if FULL else (1, 2, 3), cells=(12, 12, 12) if FULL else (6, 6, 6),
                            repeats=3 if FULL else 1)
performance.figure(timings);

## 9. The 3D wave in a helical tube and the schematics of the talk

The 1-form wave (Maxwell) in the helical tube `HelicalTube("stellarator")` with the 3D polar splines, integrated with
the leapfrog of Section 7 (`struphy.polar.homotopy.demos.waves3d`; the model version is
`examples/Maxwell/gaussian_helical_tube`). A cross-section of the initial magnetic field:

In [ ]:
import warnings

from struphy.polar.homotopy.demos import waves3d

with warnings.catch_warnings():
    warnings.simplefilter("ignore")  # the cross-sections rotate along the tube (see derham_blocks)
    wave = waves3d.Wave3D(2, 6 if FULL else 3, degree=3)
waves3d.figure_cross_section(wave, title=r"initial $B$, $\eta_3 = 0$");

In [ ]:
if FULL:
    study3d = waves3d.convergence((1, 2, 3, 4), (5, 8, 12), reference_mesh=16, final_time=0.4)
else:
    study3d = waves3d.convergence((1, 2), (3, 4), reference_mesh=5, final_time=0.1, degree=3)
waves3d.figure_convergence(study3d);

The schematics of the talk: the coordinate transformation of the helical tube and the de Rham diagrams (drawn with
`tikzfigure`, which needs LaTeX; `schematics.write_all(folder)` writes all of them):

In [ ]:
schematics.coordinate_transformation_3d();